In [9]:
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
from tqdm.auto import tqdm
from pipeline import EntityResolutionPipeline
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

print("=== STAGE 1.5: TRANSLITERATION + WIDENED UNION BLOCKING ===")

# 1. Load Data
er = EntityResolutionPipeline(sample_size=50000)
er.load_and_sample()

cand_pool = pd.concat([er.sample_s2, er.sample_s3]).drop_duplicates(subset=['entity_id']).reset_index(drop=True)
er.sample_s1 = er.sample_s1.reset_index(drop=True)

# 2. Inline Transliteration
def convert_to_latin(text):
    if not isinstance(text, str) or text == "":
        return ""
    # Converts Devanagari characters to Latin phonetics instantly
    return transliterate(text, sanscript.DEVANAGARI, sanscript.ITRANS).lower()

tqdm.pandas(desc="Transliterating S1 Names")
er.sample_s1['trans_name'] = er.sample_s1['clean_name'].progress_apply(convert_to_latin)

tqdm.pandas(desc="Transliterating Candidate Names")
cand_pool['trans_name'] = cand_pool['clean_name'].progress_apply(convert_to_latin)

# ---------------------------------------------------------
# SCHEME C: TF-IDF (Widened Parameters)
# ---------------------------------------------------------
print("\nRunning Scheme C: TF-IDF Fuzzy Name Matching...")
vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, lowercase=True)
vectorizer.fit(pd.concat([er.sample_s1['trans_name'], cand_pool['trans_name']]))

s1_tfidf = vectorizer.transform(er.sample_s1['trans_name'])
cand_tfidf = vectorizer.transform(cand_pool['trans_name'])

# WIDENED: top 50 candidates, lowering similarity threshold to 0.15
matches_matrix = sp_matmul_topn(s1_tfidf, cand_tfidf.T, top_n=50, threshold=0.15)

non_zeros = matches_matrix.nonzero()
s1_ids_array = er.sample_s1['entity_id'].values
cand_ids_array = cand_pool['entity_id'].values

tfidf_pairs = pd.DataFrame({
    's1_id': s1_ids_array[non_zeros[0]],
    'cand_id': cand_ids_array[non_zeros[1]]
})
print(f"Widened TF-IDF generated {len(tfidf_pairs):,} pairs.")

# ---------------------------------------------------------
# SCHEME A & D: Exact Name & Address (Using transliterated text)
# ---------------------------------------------------------
print("Running Exact Match Nets...")
s1_valid_names = er.sample_s1[er.sample_s1['trans_name'] != ''][['entity_id', 'trans_name']]
cand_valid_names = cand_pool[cand_pool['trans_name'] != ''][['entity_id', 'trans_name']]
exact_name_df = s1_valid_names.merge(cand_valid_names, on='trans_name')
exact_name_pairs = pd.DataFrame({'s1_id': exact_name_df['entity_id_x'], 'cand_id': exact_name_df['entity_id_y']})

s1_valid_addrs = er.sample_s1[er.sample_s1['clean_address'] != ''][['entity_id', 'clean_address']]
cand_valid_addrs = cand_pool[cand_pool['clean_address'] != ''][['entity_id', 'clean_address']]
exact_addr_df = s1_valid_addrs.merge(cand_valid_addrs, on='clean_address')
exact_addr_pairs = pd.DataFrame({'s1_id': exact_addr_df['entity_id_x'], 'cand_id': exact_addr_df['entity_id_y']})

# ---------------------------------------------------------
# UNION, DEDUPLICATE & MEASURE
# ---------------------------------------------------------
print("\nUnioning all schemes and deduplicating...")
blocked_pairs = pd.concat([tfidf_pairs, exact_name_pairs, exact_addr_pairs]).drop_duplicates(subset=['s1_id', 'cand_id'])
print(f"Total Unique Candidate Pairs: {len(blocked_pairs):,}")

print("\n=== MEASURING BLOCKING RECALL ===")
gt_pairs = set()
valid_cand_ids = set(cand_pool['entity_id'])

for _, row in tqdm(er.sample_gt.iterrows(), total=len(er.sample_gt), desc="Parsing Ground Truth"):
    s1_id = row['source1_entity_id']
    raw_matches = row['matched_entity_ids']
    if pd.notna(raw_matches):
        for m in str(raw_matches).split(','):
            clean_m = m.strip()
            if clean_m in valid_cand_ids: 
                gt_pairs.add((s1_id, clean_m))

found_pairs = set(zip(blocked_pairs['s1_id'], blocked_pairs['cand_id']))
hits = len(gt_pairs.intersection(found_pairs))
total_gt = len(gt_pairs)
recall = hits / total_gt if total_gt > 0 else 0

print(f"Ground Truth Pairs to Find: {total_gt:,}")
print(f"Pairs Successfully Blocked: {hits:,}")
print(f"FINAL BLOCKING RECALL: {recall:.4f}")
print(f"Average Candidates per S1: {len(blocked_pairs) / len(er.sample_s1):.1f}")


=== STAGE 1.5: TRANSLITERATION + WIDENED UNION BLOCKING ===
Resolving dataset paths...
Loading datasets...
Creating 50000-entity sample...
Cleaning text strings...
Data loaded, sampled, and cleaned successfully!


Transliterating S1 Names:   0%|          | 0/50000 [00:00<?, ?it/s]

Transliterating Candidate Names:   0%|          | 0/172784 [00:00<?, ?it/s]


Running Scheme C: TF-IDF Fuzzy Name Matching...
Widened TF-IDF generated 2,484,242 pairs.
Running Exact Match Nets...

Unioning all schemes and deduplicating...
Total Unique Candidate Pairs: 2,487,068

=== MEASURING BLOCKING RECALL ===


Parsing Ground Truth:   0%|          | 0/50000 [00:00<?, ?it/s]

Ground Truth Pairs to Find: 172,784
Pairs Successfully Blocked: 149,318
FINAL BLOCKING RECALL: 0.8642
Average Candidates per S1: 49.7
